In [2]:
import json
from pathlib import Path

DATA = Path("../data")
eb = json.loads((DATA / "evidence_blocks.json").read_text(encoding="utf-8"))

# The intended reference is the model's mean predicted probability on the
# training data. 03 already computed exactly that for the LIME condition of
# every model, so it is reused here for the broken SHAP conditions rather than
# reloading the models. TreeExplainer had reported the XGBoost and LightGBM
# SHAP baselines in log-odds, making every case appear above average.
lime_base = {
    (b["domain"], b["model"]): b["base_value"]
    for b in eb if b["xai"] == "LIME"
}

n = 0
for b in eb:
    if b["xai"] == "SHAP" and b["model"] in ("XGBoost", "LightGBM"):
        b["base_value"] = lime_base[(b["domain"], b["model"])]
        n += 1

(DATA / "evidence_blocks.json").write_text(
    json.dumps(eb, ensure_ascii=False, indent=1), encoding="utf-8"
)
print(f"수정한 블록: {n:,}  (기대값 1,200)")
print(lime_base)

수정한 블록: 1,200  (기대값 1,200)
{('churn', 'LightGBM'): 0.2647, ('churn', 'RandomForest'): 0.265, ('churn', 'XGBoost'): 0.2646, ('credit', 'LightGBM'): 0.2187, ('credit', 'RandomForest'): 0.2214, ('credit', 'XGBoost'): 0.2184}


In [3]:
import json
from pathlib import Path

DATA = Path("../data")
AFFECTED = ("_XGBoost_SHAP_", "_LightGBM_SHAP_")

n = 0
for p in (DATA / "generations").rglob("*.json"):
    r = json.loads(p.read_text(encoding="utf-8"))
    if r["xai"] == "SHAP" and r["model"] in ("XGBoost", "LightGBM"):
        p.unlink()
        n += 1
print(f"generations 삭제: {n:,}  (기대값 20,100)")

for folder in ["claims_cache", "judge_cache"]:
    k = 0
    for p in (DATA / folder).glob("*.json"):
        if any(a in p.name for a in AFFECTED):
            p.unlink()
            k += 1
    print(f"{folder} 삭제: {k:,}")

generations 삭제: 20,100  (기대값 20,100)
claims_cache 삭제: 20,100
judge_cache 삭제: 22,280
